# EICC GRPO Training Notebook (Theme #3.1 World Modeling)

Colab-ready notebook for the Enterprise Incident Command Center. Run cells top to bottom:
1. Clone repo
2. Install dependencies (Unsloth + TRL + peft + bitsandbytes)
3. Dry-run sanity check
4. Baseline evaluation (writes `artifacts/eval/baseline_report.json`)
5. GRPO training — **choose 5a (quick, ~1.5-2h on T4) or 5b (full, ~6-8h)**
6. Post-training compare evaluation (checkpoint vs baseline)
7. Inspect reports (`baseline_report.json`, `trained_report.json`, `policy_used`)
8. Display reward curves
9. (Optional) Two-seed reproducibility
10. (Optional) Mean ± std summary

All artifacts land in `artifacts/...` and can be downloaded from the Colab file browser.

In [ ]:
# Step 1: Clone your repo
!git clone https://github.com/anujkamaljain/OpenEnv-Customer-Support.git
%cd OpenEnv-Customer-Support

In [ ]:
# Step 2: Install training dependencies (clean path)
# If Colab asks for runtime restart, restart and rerun from Step 1.
%pip install -q -U pip setuptools wheel jedi
%pip install -q -U "pydantic>=2.12,<3" "click<8.2"
%pip install -q -U "trl==0.15.2" "transformers==4.51.3" datasets peft bitsandbytes accelerate matplotlib llm-blender mergekit
%pip install -q -U unsloth

In [ ]:
# Step 3: Verify training stack + environment (quick sanity check)
import importlib.metadata as im
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

print("unsloth:", im.version("unsloth"))
print("trl:", im.version("trl"))
print("transformers:", im.version("transformers"))

!python train.py --iterations 1 --episodes 1 --k 2 --dry-run

In [ ]:
# Step 4: Run baseline evaluation (before training)
!python evaluate.py --policy baseline --episodes-per-difficulty 5 --output-dir artifacts/eval

In [ ]:
# Step 5a: QUICK training (recommended first pass; ~1.5-2 hrs on T4)
# After this finishes, artifacts/train/trained_adapter/ will exist and Step 6 can run.
!python -u train.py --iterations 10 --episodes 15 --k 2 --eval-episodes 1 --seed 7 --max-completion-length 96 --output-dir artifacts/train

# Step 5b: FULL training (uncomment for strongest evidence; ~6-8 hrs on T4, ~2-3 hrs on V100)
# !python -u train.py --iterations 20 --episodes 30 --k 4 --eval-episodes 5 --seed 7 --max-completion-length 128 --output-dir artifacts/train

In [ ]:
# Step 6: Post-training comparison evaluation
# This runs ~8 episodes total (4 difficulties x episodes-per-difficulty on both baseline and trained side)
# and can take 5-15 minutes on T4 because each trained step calls model.generate(...).
# Use -u for unbuffered logs so progress appears in Colab without long silences.
# Reports: artifacts/eval/baseline_report.json, trained_report.json (both include policy_used).
!python -u evaluate.py --policy compare --compare-trained-policy trained_checkpoint --checkpoint-dir artifacts/train/trained_adapter --checkpoint-base-model Qwen/Qwen2.5-3B-Instruct --episodes-per-difficulty 2 --plot --output-dir artifacts/eval

In [ ]:
# Step 7: View results (includes policy_used so you can tell checkpoint vs fallback)
from pathlib import Path
import json

baseline_path = Path("artifacts/eval/baseline_report.json")
trained_path = Path("artifacts/eval/trained_report.json")
if baseline_path.exists() and trained_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    trained = json.loads(trained_path.read_text(encoding="utf-8"))
    print(f"Baseline policy_used: {baseline.get('policy_used', 'unknown')}")
    print(f"Trained  policy_used: {trained.get('policy_used', 'unknown')}")
    print(f"Baseline avg_normalized_reward: {baseline['avg_normalized_reward']:.4f}")
    print(f"Trained  avg_normalized_reward: {trained['avg_normalized_reward']:.4f}")
    print(f"Baseline avg_raw_reward: {baseline.get('avg_raw_reward', 0.0):+.4f}")
    print(f"Trained  avg_raw_reward: {trained.get('avg_raw_reward', 0.0):+.4f}")
    print("Behavior diffs:")
    for line in trained.get("behavior_examples", []):
        print(f"- {line}")
else:
    print("Run evaluation cells first.")

In [ ]:
# Step 8: Display reward curve
from pathlib import Path
from IPython.display import Image, display

curve = Path("artifacts/eval/reward_curves.png")
if curve.exists():
    display(Image(filename=str(curve)))
else:
    print("Run compare evaluation with --plot first.")

In [ ]:
# Step 9 (optional): Two-seed reproducibility run
# Use this for final submission metrics (mean +/- std)
!python train.py --iterations 10 --episodes 15 --k 2 --seed 7 --max-completion-length 128 --output-dir artifacts/train_seed7
!python train.py --iterations 10 --episodes 15 --k 2 --seed 17 --max-completion-length 128 --output-dir artifacts/train_seed17

In [ ]:
# Step 10 (optional): Summarize two-seed metrics
from pathlib import Path
import json, math

paths = [
    Path("artifacts/train_seed7/trained_report.json"),
    Path("artifacts/train_seed17/trained_report.json"),
]
vals_norm, vals_raw = [], []
for p in paths:
    if p.exists():
        obj = json.loads(p.read_text(encoding="utf-8"))
        vals_norm.append(float(obj["avg_normalized_reward"]))
        vals_raw.append(float(obj.get("avg_raw_reward", 0.0)))

if len(vals_norm) == 2:
    mean_norm = sum(vals_norm) / 2
    mean_raw = sum(vals_raw) / 2
    std_norm = math.sqrt(sum((x - mean_norm) ** 2 for x in vals_norm) / 2)
    std_raw = math.sqrt(sum((x - mean_raw) ** 2 for x in vals_raw) / 2)
    print(f"avg_normalized_reward: {mean_norm:.4f} +/- {std_norm:.4f}")
    print(f"avg_raw_reward: {mean_raw:.4f} +/- {std_raw:.4f}")
else:
    print("Run Step 9 first to produce both seed reports.")